# Getting started - Typed Decision Model

Ask typed questions about unstructured input and get calibrated answers instead of generated text: **choice** (one of your options, with per-option probabilities), **noul** (yes/no probability) and **score** (a level on your rubric). Every answer is one of the options you supplied.

This notebook deploys the Sigmodata SageMaker model package from AWS Marketplace, routes a support ticket, triages a log line and scores a review, then deletes the endpoint.

**Before you run this:**
1. Subscribe on AWS Marketplace (**Continue to Subscribe** -> **Continue to Configuration**).
2. Copy the **Model Package ARN** from that page into the next cell.
3. Run from a SageMaker Notebook / Studio session (execution role auto-detected), or set `role` manually.

Product page: https://www.sigmodata.com/products/decision-model  
Download this notebook: https://www.sigmodata.com/products/decision-model/getting-started.ipynb

In [ ]:
# Product ARN from AWS Marketplace -> Continue to Configuration.
# That is AWS's copy of the model package (unique per region). Do not use a
# seller-account package ARN.
MODEL_PACKAGE_ARN = ""
assert MODEL_PACKAGE_ARN.startswith("arn:aws:sagemaker:"), (
    "Paste the Model Package ARN from Marketplace Continue to Configuration"
)


## 1. Deploy a real-time endpoint

Persistent endpoints bill per hour while running. On `ml.g4dn.xlarge` a single question takes about 0.45 s and 12 questions in one request about 1.6 s (measured, default settings). CPU types (`ml.m5.xlarge`) take about 2.2 s per question, for low volume only. Loading the model takes several minutes. For bulk files, skip to **Batch transform**.

In [ ]:
import sagemaker
from sagemaker import get_execution_role
from sagemaker.model import ModelPackage

session = sagemaker.Session()
role = get_execution_role()

model = ModelPackage(role=role, model_package_arn=MODEL_PACKAGE_ARN, sagemaker_session=session)
predictor = model.deploy(initial_instance_count=1, instance_type="ml.g4dn.xlarge")

## 2. Ask several questions about one state

**Request:** `{"state": <text or JSON>, "questions": {"<name>": {"type": ..., "instructions": ..., "criteria": ...}}}`  
Questions about the same state share one encoding, so asking more costs little extra. The shapes follow the public Jev API.

In [ ]:
from sagemaker.serializers import JSONSerializer
from sagemaker.deserializers import JSONDeserializer

predictor.serializer = JSONSerializer()
predictor.deserializer = JSONDeserializer()

request = {
    "state": "Ticket #4411: I was charged twice for my March invoice and support has not replied in 3 days. Fix this or I am cancelling.",
    "questions": {
        "route": {"type": "choice", "instructions": "Which team should handle this ticket?",
                  "criteria": {"billing": "payments, invoices, refunds", "technical": "bugs, outages", "sales": "purchases, upgrades"}},
        "churn": {"type": "noul", "instructions": "Is the customer threatening to cancel?"},
        "urgency": {"type": "score", "instructions": "How urgent is this ticket?",
                    "criteria": ["low", "medium", "high", "critical"]},
    },
}
answers = predictor.predict(request)["answers"]
print("route:  ", answers["route"]["choice"], answers["route"]["probabilities"])
print("churn:  ", answers["churn"]["noul"])
u = answers["urgency"]
print("urgency:", u["legend"][str(u["level"])], "(expected level %.2f, confidence %.2f)" % (u["score"], u["confidence"]))

## 3. Reading the answers

- `choice.confidence` is normalised: 0 = uniform, 1 = certain, comparable across option counts. A common pattern is to act automatically above ~0.8 and send the rest to a person.
- `noul` is P(true).
- `score.level` is the most likely rubric level (act on this); `score.score` is the expected level (sort or threshold on this). `score.confidence` measures how tightly the mass clusters on the scale.
- `state` can be a JSON object; it is rendered deterministically.

In [ ]:
order = {"amount": 4999, "country": "US", "card_country": "RU", "account_age_days": 1}
print(predictor.predict({
    "state": order,
    "questions": {"hold": {"type": "noul", "instructions": "Should this order be held for manual fraud review?"}},
})["answers"])

## 4. Clean up the endpoint

Real-time endpoints keep billing until deleted.

In [ ]:
predictor.delete_endpoint()

## Batch transform (optional)

JSON Lines in S3, one request per line -> one response per line, same order. A malformed line gets `{"error": ...}` in its slot instead of failing the job. This example runs on a CPU instance (`ml.m5.xlarge`, about 2 s per question), which works in any account. For GPU speed change `instance_type` to `ml.g4dn.xlarge`; GPU transform instances need a SageMaker service quota in your account.

In [ ]:
import boto3, json

bucket = session.default_bucket()
input_key = "sigmodata-example/decision-model/requests.jsonl"
lines = [
    request,
    {"state": "ERROR payments-api: connection refused to db-primary:5432 (retry 5/5)",
     "questions": {"severity": {"type": "score", "instructions": "How severe is this log line?",
                                "criteria": ["info", "warning", "error", "page someone"]}}},
]
boto3.client("s3").put_object(Bucket=bucket, Key=input_key, Body="".join(json.dumps(l) + "\n" for l in lines).encode())
input_s3_uri = f"s3://{bucket}/{input_key}"
output_s3_uri = f"s3://{bucket}/sigmodata-example/decision-model/output/"

In [ ]:
transformer = model.transformer(
    instance_count=1, instance_type="ml.m5.xlarge", output_path=output_s3_uri,
    strategy="MultiRecord", max_payload=6, accept="application/jsonlines", assemble_with="Line",
)
transformer.transform(data=input_s3_uri, content_type="application/jsonlines", split_type="Line", wait=True)

In [ ]:
obj = boto3.client("s3").get_object(
    Bucket=bucket, Key=f"sigmodata-example/decision-model/output/requests.jsonl.out"
)
print(obj["Body"].read().decode())

## Next steps

- Product page: https://www.sigmodata.com/products/decision-model
- Support: support@sigmodata.com